In [ ]:
import pandas as pd
import yfinance as yf
import pennylane as qml
from pennylane import numpy as np
import itertools
import matplotlib.pyplot as plt
import pennylane.transforms as qtransforms
# from mitiq.zne.scaling import fold_global
# from mitiq.zne.inference import RichardsonFactory
from mitiq import zne
from cirq import DensityMatrixSimulator, depolarize
from mitiq.interface import convert_to_mitiq
from pennylane.transforms import fold_global, poly_extrapolate
from functools import partial

class CustumVQE:
    def __init__(self, gamma=1, B=2, P=1.0, p=1, stepsize=0.02, max_steps=60):
        self.N = int(2)
        self.gamma = gamma
        self.B = B
        self.P = P
        self.p = int(p)
        self.stepsize = stepsize
        self.max_steps = max_steps

        # Quantum devices (ideal and noisy)
        self.dev_ideal = qml.device("default.mixed", wires=self.N+1)
        noise_gate = qml.PhaseDamping
        noise_strength = 0.7
        self.dev_noisy = qtransforms.insert(self.dev_ideal, noise_gate, noise_strength)

        # Initialize Hamiltonian and Ansatz parameters
        self._fetch_stock_data()

        self.init_params = qml.numpy.array(np.random.rand((self.p + 1) * self.N))
        # self.init_params = np.array(np.random.rand((self.p + 1) * self.N, requires_grad=True))
        # self.adj_metric_tensor = qml.adjoint_metric_tensor(self.optimized_circuit, self.dev_noisy)
       
        # QNodes for ideal and noisy simulations
        self.ideal_qnode = qml.QNode(self.cost, self.dev_ideal, diff_method="parameter-shift")
        self.noisy_qnode = qml.QNode(self.cost, self.dev_noisy, diff_method="parameter-shift")
        self.mitigated_qnode = qtransforms.mitigate_with_zne(
            self.noisy_qnode,
            scale_factors=[1, 2, 3],
            folding=fold_global,
            extrapolate=poly_extrapolate,
            extrapolate_kwargs={'order': 2}
        )
               
        
        # Tracking costs for plotting
        self.ideal_costs = []
        self.noisy_costs = []
        self.mitigated_costs = []
        
    
    
    def _fetch_stock_data(self):
        """Fetch stock data and compute covariance matrix (Sigma) and mean returns (R)."""
        assets = "AAPL MSFT AMZN TSLA GOOG BRK-B"
        StockStartDate = '2018-01-01'
        StockEndDate = '2018-12-31'
        interval = '1d'
        df = yf.download(assets, start=StockStartDate,\
                 end=StockEndDate, interval=interval)['Adj Close']
        ret = df.pct_change().dropna()
        self.R = ret.mean() * 252
        self.Sigma = ret.cov() * 252

    def _build_hamiltonian(self):
        """Construct the problem Hamiltonian."""
        ZZ = [qml.PauliZ(i) @ qml.PauliZ(j) for i in range(int(self.N)) for j in range(i + 1, int(self.N))]
        ZZ_coeff = [0.5 * (self.gamma * self.Sigma.values[i][j] + self.P) for i in range(int(self.N)) for j in range(i + 1, int(self.N))]

        Z = [qml.PauliZ(i) for i in range(int(self.N))]
        Z_coeff = [
            -0.5 * self.gamma * (sum(self.Sigma.values[i][:])) + 0.5 * self.R[i] - 0.5 * self.P * (self.N - 2 * self.B)
            for i in range(int(self.N))
        ]
        
        C = 0.25 * self.gamma * (sum(sum(self.Sigma.values)) + np.trace(self.Sigma)) - 0.5 * sum(self.R) + \
            0.25 * self.P * (int(self.N) + (int(self.N) - 2 * self.B) ** 2)

        obs = ZZ + Z
        coeffs = ZZ_coeff + Z_coeff
        self.H = qml.Hamiltonian(coeffs, obs, grouping_type="qwc")

    def ansatz(self, params):
        """Hardware-efficient ansatz."""
        for q in range(int(self.N)):
            qml.RY(params[q], wires=q)
        for d in range(1, int(self.p) + 1):
            for q in range(int(self.N) - 1):
                qml.CNOT(wires=[q, q + 1])
            for q in range(int(self.N)):
                qml.RY(params[d * int(self.N) + q], wires=q)
    
   
    def cost(self, params):
        """Cost function."""
        self.ansatz(params)
        
        return qml.expval(self.H)
    
    def prop_circuit(self,params):
        self.ansatz(params)
        return qml.probs(wires=range(self.N))
    

    
    def optimize(self, qnode, cost_list, use_noise_mitigation):
        """Run the optimizer to minimize the cost function and track costs."""
        
        self._build_hamiltonian()
        
        # Use QNG Optimizer
        opt = qml.GradientDescentOptimizer(stepsize=self.stepsize)
        # old_cost = 9999.999999
        # Initialize parameters
        params = self.init_params

        try:
            for i in range(self.max_steps):
                if use_noise_mitigation:
                    # mitigated_result = self.optimized_circuit(params)
                    
                    params = opt.step(lambda v: qnode(v), params)
                    obj_value = qnode(params)


                else:
                            # Standard optimization without noise mitigation
                            
                            params = opt.step(qnode, params)
                            obj_value = qnode(params)

                # Track costs
                cost_list.append(obj_value)

                # Print updates every 5 steps
                if (i + 1) % 5 == 0:
                        print("Cost after step {:5d}: {: .7f}".format(i + 1, obj_value))
                        
        except Exception as e:
            print(f"Error at step {i+1}: {e}")
            raise

        return params, obj_value
    
    def probability_circuit(self, params):
        """For analyzing the optimized circuit."""
        @qml.qnode(self.dev_ideal)
        def circuit():
            self.ansatz(params)
            return qml.probs(wires=range(self.N))
        return circuit()
    def probability_noise_circuit(self, params):
        """For analyzing the optimized circuit."""
        @qml.qnode(self.dev_noisy)
        def circuit():
            self.ansatz(params)
            return qml.probs(wires=range(self.N))
        return circuit()
    def probability_mitigated_circuit(self, params):
        """For analyzing the optimized circuit."""
        @partial(
            qtransforms.mitigate_with_zne,
            scale_factors=[1., 2., 3.],
            folding=fold_global,
            extrapolate=poly_extrapolate,
            extrapolate_kwargs={'order': 2})
        @qml.qnode(self.dev_noisy)
        def circuit():
            self.ansatz(params)
            return qml.probs(wires=range(self.N))
        return circuit()
    
    def print_bitstring_probabilities(self, params):
        probabilities = self.probability_circuit(params)
        bit_strings = [format(i, '0{}b'.format(self.N)) for i in range(2 ** self.N)]
        print("Bit string probabilities:")
        for bit_string, probability in zip(bit_strings, probabilities):
            print(f"{bit_string}: {probability:.4f}")
  
    def print_all_probabilities(self, params, circuit_type):
        # Initialize dictionary to store probabilities for each circuit type
        circuit_types = ["noise", "mitigated"]
        all_probs = {}
        
        for circuit_type in circuit_types:
            if circuit_type == "noise":
                probs = self.probability_noise_circuit(params)    
            elif circuit_type == "mitigated":
                probs = self.probability_mitigated_circuit(params)
            
            # Store the probabilities in the dictionary
            all_probs[circuit_type] = probs
            
            # Calculate the number of qubits based on the probability array length
            N = int(np.log2(len(probs)))
            
            # Print the final solution and probabilities
            max_index = np.argmax(probs)
            max_prob = probs[max_index]
            # print(f"Final solution for {circuit_type}: {:0{}b}, with prob={max_prob:.5f}".format(max_index, N, max_prob))
            print("Final solution for : {:0{}b}, with prob={:.5f}".format(max_index, N, max_prob))
            for i, prob in enumerate(probs):
                print("Bit string: {:0{}b}, Probability: {:.5f}".format(i, N, prob))
        
        # Create a merged bar plot for all circuit types
        bit_strings = [f"{i:0{N}b}" for i in range(2**N)]  # Generate bit strings based on number of qubits
        x = np.arange(len(bit_strings))  # Label locations

        width = 0.25  # Width of bars
        fig, ax = plt.subplots(figsize=(10, 6))
        
        # Plot each circuit type
        for i, circuit_type in enumerate(circuit_types):
            ax.bar(x + i*width, all_probs[circuit_type], width, label=f'{circuit_type.capitalize()} Circuit')
        
        # Set labels and title
        ax.set_xlabel('Bit Strings')
        ax.set_ylabel('Probability')
        ax.set_title('Probability Distribution of Different Circuit Types')
        ax.set_xticks(x + width)
        ax.set_xticklabels(bit_strings)
        ax.legend()

        # Show the plot
        plt.show()

    def brute_force_solution(self):
        """Find the exact solution by brute force."""
        all_combinations = itertools.product([0, 1], repeat=self.N)
        ## Set predefined threshold for eigen value comparison
        E_g = 99999.99999
        for x in all_combinations:
            print(x)
            
            E = self.gamma * np.dot(x, np.dot(self.Sigma, x)) - np.dot(self.R, x) + self.P * (sum(x) - self.B) ** 2
            print(f"Option Bit String: {x} with Eigeien Value = {E:.4f}")
            if E < E_g:
                E_g = E
                sol = x
                print(f"New optimal found: {sol} with E_g = {E_g:.4f}")
        print("Exact solution:{}".format(sol))
        return sol


    def get_probabilities_and_bitstrings(self, params):
        probabilities = self.probability_circuit(params)
        N = int(np.log2(len(probabilities)))  # Assuming probabilities are for a complete set of basis states
        bit_strings = [format(i, '0{}b'.format(N)) for i in range(2**N)]
        return probabilities, bit_strings
    def save_data_to_excel(self, filename="vqe_results.xlsx"):
        """Save the collected costs during the optimization to an Excel file."""
        data = {
            "Iteration": list(range(1, self.max_steps + 1)),
            "Ideal Costs": self.ideal_costs + [None] * (self.max_steps - len(self.ideal_costs)),
            "Noisy Costs": self.noisy_costs + [None] * (self.max_steps - len(self.noisy_costs)),
            "Mitigated Costs": self.mitigated_costs + [None] * (self.max_steps - len(self.mitigated_costs)),
        }
        df = pd.DataFrame(data)
        df.to_excel(filename, index=False)
        print(f"Results saved to {filename}")
    
    def run(self):
        """Run both the ideal and noisy VQE optimization."""
        print("Running ideal VQE optimization...")
        ideal_params, ideal_result = self.optimize(self.ideal_qnode, self.ideal_costs, use_noise_mitigation=False)

        print("\nRunning noisy VQE optimization...")
        noisy_params, noisy_result = self.optimize(self.noisy_qnode, self.noisy_costs,use_noise_mitigation=False)

        print("\nRunning after the zne mitigation")
        noisy_params, mitigated_result = self.optimize(self.mitigated_qnode, self.mitigated_costs,use_noise_mitigation=True)

        print("\nSaving results to Excel...")
        self.save_data_to_excel()  # Save results to Excel

        
        print("\nPlotting the results...")
        self.plot_results()
        
        print("Finding the exact solution using brute force...")
        self.print_all_probabilities(self.init_params, circuit_type="ideal")
        self.print_all_probabilities(self.init_params, circuit_type="noise")
        self.print_all_probabilities(self.init_params, circuit_type="mitigated")
        

        
    def plot_results(self):
        """Plot the comparison of ideal and noisy optimization results with cost on the x-axis and iteration on the y-axis."""
        iterations = range(1, self.max_steps + 1)

        plt.figure(figsize=(10, 6))
        plt.plot(iterations[:len(self.ideal_costs)],self.ideal_costs, label="Ideal", color='blue')
        plt.plot(iterations[:len(self.noisy_costs)],self.noisy_costs, label="Noisy", color='red')
        plt.plot(iterations[:len(self.mitigated_costs)],self.mitigated_costs, label="Mitigated", color='green', linestyle='--')
        
        plt.title("VQE Optimization: Ideal vs Noisy vs Mitigated")
        plt.xlabel("Iteration")
        plt.ylabel("Cost (Objective Function Value)")
        plt.legend()
        plt.grid(True)
        plt.show()
        

#### RUN Project
vqe = CustumVQE()
vqe.run()
